# RFM Analysis

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/online_retail_cleaned.csv")

In [3]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [4]:
df["TotalAmount"] = df["Quantity"] * df["Price"]

In [5]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [6]:
df["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [7]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
reference_date

Timestamp('2011-12-10 12:50:00')

In [8]:
rfm = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum")
)

rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,12,77556.46
12347.0,2,8,4921.53
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


In [9]:
rfm.describe()

,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,342.280000
50%,96.000000,3.000000,867.740000
75%,380.000000,7.000000,2248.305000
max,739.000000,398.000000,580987.040000


In [10]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

In [11]:
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

In [12]:
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

In [13]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(int).astype(str)
    + rfm["F_Score"].astype(int).astype(str)
    + rfm["M_Score"].astype(int).astype(str)
)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,,,,
12346.0,326,12,77556.46,2,5,5,255
12347.0,2,8,4921.53,5,4,5,545
12348.0,75,5,2019.40,3,4,4,344
12349.0,19,4,4428.69,5,3,5,535
12350.0,310,1,334.40,2,1,2,212


In [14]:
rfm["RFM_Score"].value_counts().head(10)

RFM_Score
555    469
111    317
455    248
121    172
211    165
344    164
112    152
444    147
122    135
544    131
Name: count, dtype: int64

In [16]:
rfm["RFM_Score"] = rfm["RFM_Score"].astype(int)

In [17]:
def segment_customer(row):
    if row["RFM_Score"] >= 555:
        return "Champions"
    elif row["RFM_Score"] >= 444:
        return "Loyal Customers"
    elif row["RFM_Score"] >= 333:
        return "Potential Loyalists"
    elif row["RFM_Score"] >= 222:
        return "At Risk"
    else:
        return "Lost Customers"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
Customer ID,,,,,,,,
12346.0,326,12,77556.46,2,5,5,255,At Risk
12347.0,2,8,4921.53,5,4,5,545,Loyal Customers
12348.0,75,5,2019.40,3,4,4,344,Potential Loyalists
12349.0,19,4,4428.69,5,3,5,535,Loyal Customers
12350.0,310,1,334.40,2,1,2,212,Lost Customers


In [18]:
rfm["Segment"].value_counts()

Segment
Lost Customers         1600
Potential Loyalists    1322
Loyal Customers        1278
At Risk                1209
Champions               469
Name: count, dtype: int64

In [19]:
rfm.groupby("Segment")[["Recency", "Frequency", "Monetary"]].mean().round(2)

,Recency,Frequency,Monetary
Segment,,,
At Risk,230.30,3.28,1241.81
Champions,7.85,28.74,17367.30
Lost Customers,484.91,1.73,610.58
Loyal Customers,21.10,7.93,3342.02
Potential Loyalists,74.50,5.02,1876.07


In [20]:
rfm.to_csv("../data/processed/customer_rfm.csv")

In [21]:
rfm.shape

(5878, 8)

In [22]:
import os

os.path.exists("../data/processed/customer_rfm.csv")

True

## RFM Analysis Summary

RFM analysis was performed at the customer level using:

- **Recency:** Number of days since the customer's most recent purchase.
- **Frequency:** Number of unique invoices associated with the customer.
- **Monetary:** Total amount spent by the customer.

Customers were assigned scores from 1 to 5 for each RFM metric using quintile-based scoring.

- Recency scoring was reversed because lower recency indicates a more recent purchase.
- Frequency and Monetary scoring were assigned normally, where higher values receive higher scores.

The three scores were combined to create an RFM Score.

Customers were then grouped into five project-defined segments:
- Champions
- Loyal Customers
- Potential Loyalists
- At Risk
- Lost Customers

The final RFM dataset contains **5,878 customers** and was saved as:

`data/processed/customer_rfm.csv`